In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

In [ ]:
df = pd.read_csv('bsmote_balanced_dataset.csv')
df.head()

In [ ]:
X = df.drop('CLASS', axis=1)
y = df['CLASS']

In [ ]:
numeric_cols = ['AGE', 'WARD_encoded', 'MONTH_sin', 'MONTH_cos']
categorical_cols = [col for col in X.columns if col not in numeric_cols]

In [ ]:
scaler = StandardScaler()
X_scaled = X.copy()  # Preserve original X
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])  # Scale only numeric columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
gnb = GaussianNB()
param_grid = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=gnb,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

In [ ]:
print("Best parameters from GridSearchCV:", grid_search.best_params_)
print("Best cross-validation F1-score:", grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
round(accuracy_score(y_test, y_pred)*100,2)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Dengue (0)', 'Malaria (1)']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Dengue (0)', 'Malaria (1)'], yticklabels=['Dengue (0)', 'Malaria (1)'])
plt.title('Confusion Matrix for Gaussian Naive Bayes')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('gaussianNB_CM_BFRsmpl.png', dpi=400)
plt.show()

# **LIME Explanation**

In [ ]:
!pip install lime

In [ ]:
import lime
import lime.lime_tabular
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns,
    class_names=['Dengue (0)', 'Malaria (1)'],
    mode='classification',
    random_state=42
)

In [ ]:
instance_idx = 0
instance = X_test.iloc[instance_idx].values
lime_exp = lime_explainer.explain_instance(
    data_row=instance,
    predict_fn=best_model.predict_proba,
    num_features=len(X.columns)  # Include all features to aggregate later
)

In [ ]:
len(X.columns)

In [ ]:
instance


In [ ]:
num_instances = 10

print("The 10 instances that WILL be used:")
print(X_test.iloc[:num_instances])

In [ ]:
print("Dataset indices of the 10 instances:")
print(X_test.index[:num_instances].tolist())

In [ ]:
lime_exp.show_in_notebook()

In [ ]:
plt.figure()
lime_exp.as_pyplot_figure()
plt.title('LIME Explanation for First Instance (Logistic Regression)')
plt.tight_layout()
plt.savefig('lime_explanation_instance_0_logistic.png', dpi=400)
plt.show()

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data = X_train.values,
    feature_names = X_train.columns.tolist(),
    class_names = ['Dengue(0)', 'Malaria(1)'],   # 0 → Dengue, 1 → Malaria
    mode = 'classification'
)

In [ ]:
num_instances = 10

for i in range(num_instances):
    instance = X_test.iloc[i]

    explanation = lime_explainer.explain_instance(
        instance,
        best_model.predict_proba,
        num_features=len(X.columns)  # show all  features
    )

    # Show explanation
    explanation.show_in_notebook()

In [ ]:
num_instances = 10

for i in range(num_instances):
    print(f"\nInstance {i}")
    print("Dataset index:", X_test.index[i])
    print(X_test.iloc[i])


In [ ]:
num_instances = 10
for i in range(num_instances):
    instance = X_test.iloc[i]
    #explanation = lime_explainer.explain_instance(instance, best_model.predict_proba, num_features=5)
    explanation = lime_explainer.explain_instance(instance, best_model.predict_proba, num_features=len(X.columns))
    a = explanation.as_list()
    # print(explanation.as_list())
    print(a)
    print("\n")

In [ ]:
all_explanations = []

num_instances = 10
for i in range(num_instances):
    instance = X_test.iloc[i]
    explanation = lime_explainer.explain_instance(
        instance,
        best_model.predict_proba,
        num_features=len(X.columns)  # ask for all 22
    )
    all_explanations.append(explanation.as_list())

In [ ]:
from collections import defaultdict
import numpy as np

feature_weights = defaultdict(list)

for exp in all_explanations:
    for feature_cond, weight in exp:
        for col in X.columns:
            if col in feature_cond:
                feature_weights[col].append(abs(weight))
                break

In [ ]:
avg_weight_df = pd.DataFrame({
    "Feature": X.columns,
    "Avg_Absolute_Weight": [
        np.mean(feature_weights[col]) if len(feature_weights[col]) > 0 else 0
        for col in X.columns
    ]
}).sort_values(by="Avg_Absolute_Weight", ascending=False)

print(avg_weight_df)


In [ ]:
len(avg_weight_df)


In [ ]:
thresholds = sorted(
    avg_weight_df["Avg_Absolute_Weight"].unique()
)

print("Threshold values:")
for t in thresholds:
    print(t)



In [ ]:
len(thresholds)

In [ ]:
# Fix ONE sample index
sample_index = 0
x_sample = X_test.iloc[sample_index]

comparison = []

for t in thresholds:
    selected_features = avg_weight_df[
        avg_weight_df["Avg_Absolute_Weight"] >= t
    ]["Feature"].tolist()

    comparison.append({
        "Threshold": t,
        "Num_Selected_Features": len(selected_features),
        "Selected_Features": selected_features
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df)


In [ ]:
all_samples_comparison = []

num_instances = 10

for i in range(num_instances):
    sample_row = {
        "Sample_Index": i,
        "Threshold_Comparison": []
    }

    for t in thresholds:
        selected_features = avg_weight_df[
            avg_weight_df["Avg_Absolute_Weight"] >= t
        ]["Feature"].tolist()

        sample_row["Threshold_Comparison"].append({
            "Threshold": t,
            "Num_Selected_Features": len(selected_features),
            "Selected_Features": selected_features
        })

    all_samples_comparison.append(sample_row)


In [ ]:
for sample in all_samples_comparison:
    print(f"\n================ Sample {sample['Sample_Index']} ================")

    for item in sample["Threshold_Comparison"]:
        print(f"Threshold: {item['Threshold']}")
        print(f"Num Selected Features: {item['Num_Selected_Features']}")
        print(f"Selected Features: {item['Selected_Features']}")
        print("-" * 60)

In [ ]:
rows = []

for sample in all_samples_comparison:
    for item in sample["Threshold_Comparison"]:
        rows.append({
            "Sample_Index": sample["Sample_Index"],
            "Threshold": item["Threshold"],
            "Num_Selected_Features": item["Num_Selected_Features"],
            "Selected_Features": item["Selected_Features"]
        })

comparison_df = pd.DataFrame(rows)
print(comparison_df)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score



In [ ]:
for _, row in comparison_df.iterrows():

    sample_index = row["Sample_Index"]
    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n===================================")
    print(f"Sample {sample_index}")
    print(f"Threshold = {threshold}")
    print("Selected Features:", selected_features)

    # Safety check
    if len(selected_features) == 0:
        print("No features selected. Skipping.")
        continue

In [ ]:
for _, row in comparison_df.iterrows():

    sample_index = row["Sample_Index"]
    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n===================================")
    print(f"Sample {sample_index}")
    print(f"Threshold = {threshold}")
    print("Selected Features:", selected_features)

    # Safety check
    if len(selected_features) == 0:
        print("No features selected. Skipping.")
        continue

In [ ]:
sample0_df = comparison_df[comparison_df['Sample_Index'] == 0].copy()
final_results_gnb = []

for _, row in sample0_df.iterrows():

    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n========================================")
    print(f"Threshold = {threshold}")
    print("Selected Features:", selected_features)
    print("Number of Features:", len(selected_features))

    # -------------------------------
    # 1. Create reduced datasets
    # -------------------------------
    X_train_selected = X_train[selected_features]
    X_test_selected  = X_test[selected_features]

    print("\nX_train_selected.head()")
    display(X_train_selected.head())

    # -------------------------------
    # 2. StratifiedKFold + GridSearch (GNB)
    # -------------------------------
    skf_gnb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    gnb = GaussianNB()

    param_grid_gnb = {
        'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3]
    }

    grid_search_gnb = GridSearchCV(
        estimator=gnb,
        param_grid=param_grid_gnb,
        cv=skf_gnb,
        scoring='f1',
        n_jobs=-1
    )

    grid_search_gnb.fit(X_train_selected, y_train)

    print("Best GNB Parameters:", grid_search_gnb.best_params_)
    print("Best CV F1-score:", grid_search_gnb.best_score_)

    # -------------------------------
    # 3. Best GNB model
    # -------------------------------
    best_model_gnb = grid_search_gnb.best_estimator_
    best_model_gnb.fit(X_train_selected, y_train)

    # -------------------------------
    # 4. Test on FIRST 10 rows
    # -------------------------------
    X_test_subset = X_test_selected.iloc[:10]
    y_test_subset = y_test.iloc[:10]

    y_pred_subset = best_model_gnb.predict(X_test_subset)

    acc_subset = round(accuracy_score(y_test_subset, y_pred_subset) * 100, 2)
    print("Subset Accuracy (%):", acc_subset)

    print("\nClassification Report (Subset):")
    print(classification_report(
        y_test_subset, y_pred_subset,
        target_names=['Dengue (0)', 'Malaria (1)']
    ))

    # -------------------------------
    # 5. Confusion Matrix (BLUE-WHITE)
    # -------------------------------
    cm = confusion_matrix(y_test_subset, y_pred_subset)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Dengue (0)', 'Malaria (1)'],
        yticklabels=['Dengue (0)', 'Malaria (1)']
    )
    plt.title(f'GaussianNB Confusion Matrix (Threshold={round(threshold,4)})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # -------------------------------
    # 6. Store results
    # -------------------------------
    final_results_gnb.append({
        "Threshold": threshold,
        "Num_Features": len(selected_features),
        "Subset_Accuracy": acc_subset,
        "CV_F1": grid_search_gnb.best_score_,
        "Selected_Features": selected_features
    })

# **SHAP Explanations**

In [ ]:
# import shap

# explainer = shap.LinearExplainer(best_model, X_train, feature_names=X.columns)

import shap

explainer = shap.Explainer(best_model.predict_proba, X_train)
# shap_values = explainer(X_train)

In [ ]:
X_test_subset = X_test.iloc[:50]
X_test_subset.head()

In [ ]:
shap_values = explainer(X_test_subset)

In [ ]:
shap_values[0]

#Beeswarm Summary Plot

In [ ]:
shap.plots.beeswarm(shap_values[..., 1])

In [ ]:
shap.plots.beeswarm(shap_values[..., 0])

In [ ]:
print(type(shap_values))
print(shap_values.shape)

In [ ]:
print("Shape of X_test_subset:", X_test_subset.shape)
print("Shape of shap_values:", shap_values.shape)

# Bar plot

In [ ]:
shap.plots.bar(shap_values[..., 1])

In [ ]:
shap.plots.bar(shap_values[..., 0])

In [ ]:
shap_values[..., 1]

In [ ]:
shap_values[..., 0]

In [ ]:

import numpy as np

shap_vals = shap_values.values
mean_abs_shap_avg = np.mean(np.abs(shap_vals), axis=(0, 2))  # mean over samples and classes

In [ ]:
mean_abs_shap_avg

In [ ]:
normalized_shap = mean_abs_shap_avg / np.sum(mean_abs_shap_avg)
normalized_shap

In [ ]:
# Create a DataFrame
feature_names = shap_values.feature_names
shap_df = pd.DataFrame({
    'Feature': feature_names,
    'MeanAbsSHAP': mean_abs_shap_avg,
    'NormalizedSHAP': normalized_shap
})
shap_df

In [ ]:
# Rank features
shap_df['Rank'] = shap_df['MeanAbsSHAP'].rank(ascending=False).astype(int)
shap_df['Rank']

In [ ]:
# Sort
shap_df = shap_df.sort_values(by='MeanAbsSHAP', ascending=False)
print(shap_df)

In [ ]:
import numpy as np

shap_thresholds = sorted(np.unique(mean_abs_shap_avg))

print("SHAP Thresholds:")
for t in shap_thresholds:
    print(t)


In [ ]:
len(shap_thresholds)

In [ ]:
X_train.columns.tolist()

In [ ]:
feature_names = X_train.columns.tolist()

shap_results = []

for t in shap_thresholds:

    t_clean = float(t)   # clean threshold

    # Compare threshold with GLOBAL SHAP importance
    mask = mean_abs_shap_avg >= t

    # 🔑 FIX: force Python str for feature names
    selected_features = [str(f) for f in np.array(feature_names)[mask]]

    # Safety check
    if len(selected_features) == 0:
        continue

    shap_results.append({
        "Threshold": t_clean,
        "Num_Selected_Features": len(selected_features),
        "Selected_Features": selected_features
    })

    print("\n===================================")
    print("Threshold:", t_clean)
    print("Selected Features:", selected_features)
    print("Number of Features:", len(selected_features))


In [ ]:
print("\nIndex → Feature → Mean |SHAP|")
print("-" * 45)

for i, (fname, shap_val) in enumerate(zip(feature_names, mean_abs_shap_avg)):
    print(f"{i:2d} → {fname:20s} → {shap_val:.6f}")


In [ ]:
shap_feature_selection_df = pd.DataFrame(shap_results)
shap_feature_selection_df

In [ ]:
final_results_shap_gnb = []

for _, row in shap_feature_selection_df.iterrows():

    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n===================================")
    print("TRAINING FOR SHAP THRESHOLD:", threshold)
    print("Selected Features:", selected_features)
    print("Number of Features:", len(selected_features))

    # ----------------------------
    # 1. Reduce datasets
    # ----------------------------
    X_train_selected = X_train[selected_features]
    X_test_selected  = X_test[selected_features]

    # ----------------------------
    # 2. StratifiedKFold + GridSearch (GaussianNB)
    # ----------------------------
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    gnb = GaussianNB()

    param_grid = {
        'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3]
    }

    grid = GridSearchCV(
        estimator=gnb,
        param_grid=param_grid,
        cv=skf,
        scoring='f1',
        n_jobs=-1
    )

    grid.fit(X_train_selected, y_train)

    print("Best Params:", grid.best_params_)
    print("CV Best F1:", grid.best_score_)

    best_model = grid.best_estimator_

    # ----------------------------
    # 3. TEST ON FIRST 50 SAMPLES
    # ----------------------------
    X_test_subset = X_test_selected.iloc[:50]
    y_test_subset = y_test.iloc[:50]

    y_pred = best_model.predict(X_test_subset)

    acc = accuracy_score(y_test_subset, y_pred)
    f1  = f1_score(y_test_subset, y_pred)

    print("Subset Accuracy:", round(acc * 100, 2))
    print("Subset F1:", round(f1, 4))

    print("\nClassification Report:")
    print(classification_report(
        y_test_subset,
        y_pred,
        target_names=['Dengue (0)', 'Malaria (1)']
    ))

    # ----------------------------
    # 4. Confusion Matrix (BLUE-WHITE)
    # ----------------------------
    cm = confusion_matrix(y_test_subset, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Dengue (0)', 'Malaria (1)'],
        yticklabels=['Dengue (0)', 'Malaria (1)']
    )
    plt.title(f'GNB SHAP Confusion Matrix (Threshold={round(threshold,4)})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # 5. Store results
    # ----------------------------
    final_results_shap_gnb.append({
        "Threshold": threshold,
        "Num_Features": len(selected_features),
        "Accuracy": acc,
        "F1_Score": f1,
        "Selected_Features": selected_features
    })
